# Bash 链
本 Notebook 展示了如何使用 LLM 和 bash 进程来执行简单的文件系统命令。

In [1]:
from langchain_experimental.llm_bash.base import LLMBashChain
from langchain_openai import OpenAI

llm = OpenAI(temperature=0)

text = "Please write a bash script that prints 'Hello World' to the console."

bash_chain = LLMBashChain.from_llm(llm, verbose=True)

bash_chain.invoke(text)



> Entering new LLMBashChain chain...
Please write a bash script that prints 'Hello World' to the console.

```bash
echo "Hello World"
```
Code: ['echo "Hello World"']
Answer: Hello World

> Finished chain.


'Hello World\n'

## 自定义提示

您也可以自定义所使用的提示。以下是一个避免使用 'echo' 工具的示例提示：

In [2]:
from langchain.prompts.prompt import PromptTemplate
from langchain_experimental.llm_bash.prompt import BashOutputParser

_PROMPT_TEMPLATE = """If someone asks you to perform a task, your job is to come up with a series of bash commands that will perform the task. There is no need to put "#!/bin/bash" in your answer. Make sure to reason step by step, using this format:
Question: "copy the files in the directory named 'target' into a new directory at the same level as target called 'myNewDirectory'"
I need to take the following actions:
- List all files in the directory
- Create a new directory
- Copy the files from the first directory into the second directory
```bash
ls
mkdir myNewDirectory
cp -r target/* myNewDirectory
```

Do not use 'echo' when writing the script.

That is the format. Begin!
Question: {question}"""

PROMPT = PromptTemplate(
    input_variables=["question"],
    template=_PROMPT_TEMPLATE,
    output_parser=BashOutputParser(),
)

In [3]:
bash_chain = LLMBashChain.from_llm(llm, prompt=PROMPT, verbose=True)

text = "Please write a bash script that prints 'Hello World' to the console."

bash_chain.invoke(text)



> Entering new LLMBashChain chain...
Please write a bash script that prints 'Hello World' to the console.

```bash
printf "Hello World\n"
```
Code: ['printf "Hello World\\n"']
Answer: Hello World

> Finished chain.


'Hello World\n'

## 持久化终端

默认情况下，每次调用链时，它都会在一个单独的子进程中运行。通过实例化一个持久化的 bash 进程可以改变这种行为。

In [4]:
from langchain_experimental.llm_bash.bash import BashProcess

persistent_process = BashProcess(persistent=True)
bash_chain = LLMBashChain.from_llm(llm, bash_process=persistent_process, verbose=True)

text = "List the current directory then move up a level."

bash_chain.invoke(text)



> Entering new LLMBashChain chain...
List the current directory then move up a level.

```bash
ls
cd ..
```
Code: ['ls', 'cd ..']
Answer: cpal.ipynb  llm_bash.ipynb  llm_symbolic_math.ipynb
index.mdx   llm_math.ipynb  pal.ipynb
> Finished chain.


'cpal.ipynb  llm_bash.ipynb  llm_symbolic_math.ipynb\r\nindex.mdx   llm_math.ipynb  pal.ipynb'

In [5]:
# Run the same command again and see that the state is maintained between calls
bash_chain.invoke(text)



> Entering new LLMBashChain chain...
List the current directory then move up a level.

```bash
ls
cd ..
```
Code: ['ls', 'cd ..']
Answer: _category_.yml	data_generation.ipynb		   self_check
agents		graph
code_writing	learned_prompt_optimization.ipynb
> Finished chain.


'_category_.yml\tdata_generation.ipynb\t\t   self_check\r\nagents\t\tgraph\r\ncode_writing\tlearned_prompt_optimization.ipynb'